In [2]:
import torch
from monai.data import ImageDataset, DataLoader
from monai.transforms import EnsureChannelFirst, Compose, Rand3DElastic, Resize, NormalizeIntensity, RandShiftIntensity
import monai
from torch.utils.tensorboard import SummaryWriter
import json
import os
from collections import OrderedDict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms from the MONAI tutorial
train_transforms = Compose([NormalizeIntensity(nonzero=True, channel_wise=True), 
                            EnsureChannelFirst(), 
                            Resize((96, 96, 96)),
                            Rand3DElastic(prob=0.5, sigma_range=(6, 8), magnitude_range=(10, 50),spatial_size=(96, 96, 96), padding_mode="border"),
                            RandShiftIntensity(prob=0.5, offsets=0.10)
])

val_transforms = Compose([NormalizeIntensity(nonzero=True, channel_wise=True), EnsureChannelFirst(), Resize((96, 96, 96))])

## 1. Load the locked-in splits
load_path = os.path.expanduser('~/Desktop/brain-math/deeplearn/GLM/GLM_kfold_splits.json')
with open(load_path, 'r') as f:
    saved_splits = json.load(f)

# 2. Select which fold you want to train right now
current_fold = "fold_1"  # Change this to "fold_2", "fold_3", etc., when ready
print(f"Loading data for {current_fold}...")

fold_data = saved_splits[current_fold]
train_images = fold_data["train_images"]
val_images = fold_data["val_images"]

# 3. Convert the saved integer labels (0 or 1) back into one-hot tensors for MONAI
train_labels = torch.as_tensor(fold_data["train_labels"], dtype=torch.long)
val_labels = torch.as_tensor(fold_data["val_labels"], dtype=torch.long)

# 4. Create your MONAI Datasets
train_ds = ImageDataset(image_files=train_images, labels=train_labels, transform=train_transforms)
val_ds = ImageDataset(image_files=val_images, labels=val_labels, transform=val_transforms)

# 5. Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=8, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=8, num_workers=8, pin_memory=torch.cuda.is_available())

model = monai.networks.nets.ViT(
    in_channels=1,
    img_size=(96, 96, 96),
    patch_size=(16, 16, 16), # Slices the 96x96x96 brain into 216 individual cubes
    proj_type='conv',        # Uses 3D convolutions to learn where each cube belongs in space
    classification=True,     # Forces the model to output a class label, not a segmentation mask
    num_classes=2            # Math Difficulty vs. Control
).to(device)

loss_function = torch.nn.CrossEntropyLoss()

# Replace your current optimizer with AdamW and add weight_decay
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)



# start a typical PyTorch training
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
writer = SummaryWriter(log_dir="runs/ViT_test")
max_epochs = 150

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

patience = 30            # Stop after 20 validation checks without improvement
patience_counter = 0     # Tracks how long we've gone without a new best score

for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0

    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs[0], labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_ds) // train_loader.batch_size
        print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
        writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")


    model.eval()

    num_correct = 0.0
    metric_count = 0
    val_loss_sum = 0.0
    for val_data in val_loader:
        val_images, val_labels = val_data[0].to(device), val_data[1].to(device)
        with torch.no_grad():
            val_outputs = model(val_images)
            logits = val_outputs[0]
            val_loss = loss_function(logits, val_labels)
            val_loss_sum += val_loss.item()
            value = torch.eq(logits.argmax(dim=1), val_labels)
            metric_count += len(value)
            num_correct += value.sum().item()

    metric = num_correct / metric_count
    metric_values.append(metric)

    avg_val_loss = val_loss_sum / len(val_loader)

    if metric > best_metric:
        best_metric = metric
        best_metric_epoch = epoch + 1
        patience_counter = 0  # reset patience if it improves
        torch.save(model.state_dict(), "best_ViT_classification3d_array.pth")
        print("saved new best metric model")
    else:
        patience_counter += 1 # increment patience if it failed to improve
        print(f"No improvement. Patience: {patience_counter}/{patience}")

    print(f"Current epoch: {epoch+1} current accuracy: {metric:.4f} ")
    print(f"Best accuracy: {best_metric:.4f} at epoch {best_metric_epoch}")
    writer.add_scalar("val_accuracy", metric, epoch + 1)

    scheduler.step()

    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch + 1}!")
        print(f"Validation accuracy hasn't improved in {patience} epochs.")
        break

print(f"Training completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
writer.close()

Loading data for fold_1...
----------
epoch 1/150
1/24, train_loss: 0.9998
2/24, train_loss: 1.4676
3/24, train_loss: 1.3554
4/24, train_loss: 0.6273
5/24, train_loss: 0.8345
6/24, train_loss: 0.6568
7/24, train_loss: 0.7948
8/24, train_loss: 1.1274
9/24, train_loss: 0.6301
10/24, train_loss: 0.8232
11/24, train_loss: 0.6985
12/24, train_loss: 0.8389
13/24, train_loss: 0.6931
14/24, train_loss: 0.6253
15/24, train_loss: 0.8480
16/24, train_loss: 0.6931
17/24, train_loss: 0.6931
18/24, train_loss: 0.6931
19/24, train_loss: 0.7211
20/24, train_loss: 0.6339
21/24, train_loss: 0.6931
22/24, train_loss: 0.6919
23/24, train_loss: 0.6931
24/24, train_loss: 0.6952
epoch 1 average loss: 0.8012
saved new best metric model
Current epoch: 1 current accuracy: 0.3830 
Best accuracy: 0.3830 at epoch 1
----------
epoch 2/150
1/24, train_loss: 0.6225
2/24, train_loss: 0.7986
3/24, train_loss: 0.6931
4/24, train_loss: 0.6931
5/24, train_loss: 0.6931
6/24, train_loss: 0.6931
7/24, train_loss: 0.6931
8/24

KeyboardInterrupt: 

# Plans

## Plan A： Lightweight Classifier+Standard Normalization+No Pre Training

In [ ]:
import random
import numpy as np
import torch
import monai
 
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    monai.utils.set_determinism(seed=seed)
 
set_seed(42)
 
import torch
import os
import json
from monai.data import ImageDataset, DataLoader
from monai.transforms import (
    Compose, EnsureChannelFirst, Resize,
    ScaleIntensity, NormalizeIntensity,
    RandAffine, RandGaussianNoise
)
from torch.utils.tensorboard import SummaryWriter
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
# ------------------- Enhancement+Normalization -------------------
train_transforms = Compose([
    EnsureChannelFirst(),
    Resize((96, 96, 96)),
    ScaleIntensity(minv=0.0, maxv=1.0),
    NormalizeIntensity(nonzero=True),
    RandAffine(prob=0.2, rotate_range=(-2,2), translate_range=0.02),
    RandGaussianNoise(prob=0.1, std=0.01),
])
 
val_transforms = Compose([
    EnsureChannelFirst(),
    Resize((96, 96, 96)),
    ScaleIntensity(minv=0.0, maxv=1.0),
    NormalizeIntensity(nonzero=True),
])
 
## 1. Load the locked-in splits
load_path = os.path.expanduser('~/Desktop/brain-math/GLM_kfold_splits.json')
with open(load_path, 'r') as f:
    saved_splits = json.load(f)
 
current_fold = "fold_1"
fold_data = saved_splits[current_fold]
 
train_images = fold_data["train_images"]
val_images = fold_data["val_images"]
train_labels = torch.as_tensor(fold_data["train_labels"], dtype=torch.long)
val_labels = torch.as_tensor(fold_data["val_labels"], dtype=torch.long)
 
train_ds = ImageDataset(image_files=train_images, labels=train_labels, transform=train_transforms)
val_ds = ImageDataset(image_files=val_images, labels=val_labels, transform=val_transforms)
 
batch_size = 4
num_workers = 2
 
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
 
# ------------------- Model: Lightweight Classifier (without pre training) -------------------
model = monai.networks.nets.Classifier(
    in_shape=(1, 96, 96, 96),  # (channels, D, H, W)
    classes=2,                 # Replaces out_classes
    channels=(16, 32, 64),
    strides=(2, 2)             # Length should typically be len(channels) - 1
).to(device)
 
loss_function = torch.nn.CrossEntropyLoss()
 
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
#Learning rate decay: The validation set automatically decreases without increasing
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6)
 
# ------------------- training -------------------
best_metric = -1
patience = 10
patience_counter = 0
max_epochs = 80
writer = SummaryWriter()
 
for epoch in range(max_epochs):
    print("-" * 30)
    print(f"Epoch {epoch+1}/{max_epochs}")
    model.train()
    train_loss = 0
    step = 0
 
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_function(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        step += 1
 
    avg_train_loss = train_loss / step
    print(f"Train loss: {avg_train_loss:.4f}")
 
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            val_loss += loss_function(out, y).item()
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += len(y)
 
    val_acc = correct / total
    scheduler.step(val_acc)
    print(f"Val acc: {val_acc:.4f}")
 
    if val_acc > best_metric:
        best_metric = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_classifier.pth")
        print("saved best metric model")
    else:
        patience_counter += 1
        print(f"No improvement: {patience_counter}/{patience}")
 
    if patience_counter >= patience:
        print("\nEarly stopping")
        break
 
print(f"Best accuracy: {best_metric:.4f}")
writer.close()
 
 

------------------------------
Epoch 1/80


/opt/anaconda3/envs/fmriprep/lib/python3.10/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Train loss: 1.8005
Val acc: 0.6809
saved best metric model
------------------------------
Epoch 2/80
Train loss: 0.5569
Val acc: 0.5319
No improvement: 1/10
------------------------------
Epoch 3/80
Train loss: 0.5056
Val acc: 0.4894
No improvement: 2/10
------------------------------
Epoch 4/80
Train loss: 0.2929
Val acc: 0.5745
No improvement: 3/10
------------------------------
Epoch 5/80
Train loss: 0.3284
Val acc: 0.5745
No improvement: 4/10
------------------------------
Epoch 6/80
Train loss: 0.3404
Val acc: 0.5319
No improvement: 5/10
------------------------------
Epoch 7/80
Train loss: 0.3315
Val acc: 0.5106
No improvement: 6/10
------------------------------
Epoch 8/80
Train loss: 0.2206
Val acc: 0.5319
No improvement: 7/10
------------------------------
Epoch 9/80
Train loss: 0.3425
Val acc: 0.5319
No improvement: 8/10
------------------------------
Epoch 10/80
Train loss: 0.2612
Val acc: 0.4255
No improvement: 9/10
------------------------------
Epoch 11/80
Train loss: 0.2

## Plan B: DenseNet121+pre training+normalization

In [ ]:
import random
import numpy as np
import torch
import monai
 
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    monai.utils.set_determinism(seed=seed)
 
set_seed(42)
 
import torch
import os
import json
from monai.data import ImageDataset, DataLoader
from monai.transforms import (
    Compose, EnsureChannelFirst, Resize,
    ScaleIntensity, NormalizeIntensity,
    RandAffine, RandGaussianNoise
)
from torch.utils.tensorboard import SummaryWriter
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
#  ------------------- Enhancement+Normalization -------------------
train_transforms = Compose([
    EnsureChannelFirst(),
    Resize((96, 96, 96)),
    ScaleIntensity(minv=0.0, maxv=1.0),
    NormalizeIntensity(nonzero=True),
    RandAffine(prob=0.2, rotate_range=(-2,2), translate_range=0.02),
    RandGaussianNoise(prob=0.1, std=0.01),
])
 
val_transforms = Compose([
    EnsureChannelFirst(),
    Resize((96, 96, 96)),
    ScaleIntensity(minv=0.0, maxv=1.0),
    NormalizeIntensity(nonzero=True),
])
 
## 1. Load the locked-in splits
load_path = os.path.expanduser('~/Desktop/brain-math/GLM_kfold_splits.json')
with open(load_path, 'r') as f:
    saved_splits = json.load(f)
 
current_fold = "fold_1"
fold_data = saved_splits[current_fold]
 
train_images = fold_data["train_images"]
val_images = fold_data["val_images"]
train_labels = torch.as_tensor(fold_data["train_labels"], dtype=torch.long)
val_labels = torch.as_tensor(fold_data["val_labels"], dtype=torch.long)
 
train_ds = ImageDataset(image_files=train_images, labels=train_labels, transform=train_transforms)
val_ds = ImageDataset(image_files=val_images, labels=val_labels, transform=val_transforms)
 
batch_size = 4
num_workers = 2
 
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
 
# ------------------- Model：pretrained DenseNet121 -------------------
model = monai.networks.nets.DenseNet121(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    pretrained=False ##
).to(device)
 
loss_function = torch.nn.CrossEntropyLoss()
 
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6)
 
# ------------------- training -------------------
best_metric = -1
patience = 10
patience_counter = 0
max_epochs = 80
writer = SummaryWriter()
 
for epoch in range(max_epochs):
    print("-" * 30)
    print(f"Epoch {epoch+1}/{max_epochs}")
    model.train()
    train_loss = 0
    step = 0
 
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_function(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        step += 1
 
    avg_train_loss = train_loss / step
    print(f"Train loss: {avg_train_loss:.4f}")
 
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            val_loss += loss_function(out, y).item()
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += len(y)
 
    val_acc = correct / total
    scheduler.step(val_acc)
    print(f"Val acc: {val_acc:.4f}")
 
    if val_acc > best_metric:
        best_metric = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_pretrained_densenet.pth")
        print("saved best metric model")
    else:
        patience_counter += 1
        print(f"No improvement: {patience_counter}/{patience}")
 
    if patience_counter >= patience:
        print("\nEarly stopping")
        break
 
print(f"Best accuracy: {best_metric:.4f}")
writer.close()

------------------------------
Epoch 1/80


/opt/anaconda3/envs/fmriprep/lib/python3.10/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Train loss: 0.7045
Val acc: 0.3404
saved best metric model
------------------------------
Epoch 2/80
Train loss: 0.6859
Val acc: 0.3830
saved best metric model
------------------------------
Epoch 3/80
Train loss: 0.6848
Val acc: 0.3404
No improvement: 1/10
------------------------------
Epoch 4/80
Train loss: 0.6514
Val acc: 0.3404
No improvement: 2/10
------------------------------
Epoch 5/80
Train loss: 0.6307
Val acc: 0.6809
saved best metric model
------------------------------
Epoch 6/80
Train loss: 0.6253
Val acc: 0.4255
No improvement: 1/10
------------------------------
Epoch 7/80
Train loss: 0.5801
Val acc: 0.3404
No improvement: 2/10
------------------------------
Epoch 8/80
Train loss: 0.5393
Val acc: 0.3404
No improvement: 3/10
------------------------------
Epoch 9/80
Train loss: 0.6032
Val acc: 0.3617
No improvement: 4/10
------------------------------
Epoch 10/80
Train loss: 0.4368
Val acc: 0.6596
No improvement: 5/10
------------------------------
Epoch 11/80
Train los